# Notebook 7 — Binning & Discretization

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
customers[["monthly_charges","tenure_months"]].describe()

## What is Binning, and Why?

Binning converts a **continuous** numeric variable into a **categorical** one by
grouping values into ranges ("bins"). It trades some granularity for:

- **Robustness to outliers** — an outlier still falls into the "high" bin, rather than
  distorting a linear model's coefficient
- **Interpretability** — "Income Group: High" is easier for a business stakeholder to
  reason about than a raw continuous coefficient
- **Capturing non-linear/threshold effects** — real-world relationships (e.g. risk
  jumping sharply after a specific tenure threshold) aren't always linear, and binning
  lets a simple model capture that

## 1. Equal-Width Binning

Splits the range into bins of equal size, regardless of how many observations fall
into each. **When to use:** when the actual numeric range matters more than balancing
sample sizes (e.g. defined age brackets: 0-18, 19-35, ...).

In [ ]:
customers["tenure_bucket_equal_width"] = pd.cut(
    customers["tenure_months"], bins=4,
    labels=["New","Early","Established","Loyal"]
)
customers["tenure_bucket_equal_width"].value_counts().sort_index()

## 2. Equal-Frequency (Quantile) Binning

Splits so each bin has roughly the same *number of observations*. **When to use:**
when you want balanced bin sizes for modeling (avoids one bin dominating), common for
skewed monetary columns like `monthly_charges`.

In [ ]:
customers["charge_quartile"] = pd.qcut(
    customers["monthly_charges"], q=4,
    labels=["Q1 (Low)","Q2","Q3","Q4 (High)"]
)
customers["charge_quartile"].value_counts().sort_index()

## 3. Domain-Based Binning

**When to use:** whenever the business already has meaningful, established thresholds
— these almost always outperform statistically-derived bins because they encode real
operational knowledge (e.g. telecom risk tiers used by the retention team).

In [ ]:
def risk_tier(row):
    if row["tenure_months"] < 6:
        return "High Risk (New)"
    elif row["contract"] == "Month-to-month" and row["monthly_charges"] > 80:
        return "High Risk (Price-sensitive, no lock-in)"
    elif row["tenure_months"] > 36:
        return "Low Risk (Loyal)"
    else:
        return "Medium Risk"

customers["business_risk_tier"] = customers.apply(risk_tier, axis=1)
customers["business_risk_tier"].value_counts()

In [ ]:
churn_by_tier = (
    customers.assign(churn_binary=(customers["churn"]=="Yes").astype(int))
    .groupby("business_risk_tier")["churn_binary"].mean().sort_values(ascending=False)
)
churn_by_tier

This domain-derived tier separates churn rate far more cleanly than any single raw
numeric column — exactly why domain-based binning is often worth the manual effort.

## Age Groups / Income Groups / Risk Categories — Generalized Pattern

The same `pd.cut` / rule-based pattern generalizes directly to:
- **Age Groups**: `pd.cut(age, bins=[0,18,35,50,65,120], labels=[...])`
- **Income Groups**: quantile or domain-defined brackets (e.g. tax brackets)
- **Risk Categories**: exactly the `business_risk_tier` pattern shown above

## Advantages and Limitations of Binning

**Advantages:**
- Robust to outliers and non-linear relationships
- Improves interpretability for stakeholders and simple models
- Can encode genuine business/domain thresholds directly

**Limitations:**
- **Information loss** — two very different values in the same bin become
  indistinguishable to the model
- **Bin boundary sensitivity** — a customer at month 5 vs month 6 can land in
  different bins despite being nearly identical
- Poor choice of bin count/edges can *hide* real signal rather than reveal it
- Tree-based models can already learn effective splits on their own, making manual
  binning less necessary (though still useful for interpretability)

## Summary — Feature Justification

| Feature | Source | Logic | Leakage Risk | Decision |
|---|---|---|---|---|
| `business_risk_tier` | tenure_months, contract, monthly_charges | rule-based domain bucketing | None | **Retain** — clear separation of churn rate |
| `charge_quartile` | monthly_charges | quantile binning | None | **Retain** as a categorical companion to the raw numeric column |
| `tenure_bucket_equal_width` | tenure_months | equal-width binning | None | **Needs further analysis** — quantile version may separate churn better |